# Loading exported estimators directly

The public inference surface consists of `load_skops`, `load_default`, and `load_torch`. Each function returns the underlying sklearn or skorch estimator, so prediction uses that estimator's native `predict` and `predict_proba` API. Only load artifacts you trust.

Like the other example notebooks, every section uses the real AGN-versus-star-forming Cloudy grids under `data/`. Each section owns its imports, temporary output workspace, data loading, and trainer configuration. Within a section, run the short code cells from top to bottom.

## SimpleTrainer and `load_skops`

This section uses the same silver-level CSV files and YAML configuration as `simpletrainer_examples.ipynb`. The two source files contain Cloudy AGN models encoded as class 0 and POPSTAR models encoded as class 1.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import torch
import yaml
from torch.utils.data import random_split

from GalaxySpectrumClassifier import SimpleTrainer, TabularDataset, load_skops


def find_project_root():
    """Find the repository whether the kernel starts in root or notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "configs").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root")


simple_root = find_project_root()
with (simple_root / "configs/binary_classsifier_simple_example.yaml").open() as stream:
    simple_config = yaml.safe_load(stream)

# Resolve the notebook-relative config path explicitly for any kernel cwd.
simple_config["dataset"]["path"] = str(simple_root / "data/silver/default")
simple_dataset = TabularDataset.from_config(simple_config["dataset"])
simple_train, simple_test = random_split(
    simple_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

### Train and export

The trainer is built directly from the same calibrated-random-forest configuration used by `simpletrainer_examples.ipynb`. Only its output path is redirected to a temporary workspace so the cell can be rerun safely.

In [ ]:
simple_workspace = Path(tempfile.mkdtemp(prefix="gsc-simple-inference-"))
simple_trainer_config = simple_config["trainer"]
simple_trainer_config["output_path"] = str(simple_workspace / "training")

simple_trainer = SimpleTrainer.from_config(simple_trainer_config)
simple_trainer.fit(simple_train)

# A standalone SimpleTrainer export is one fitted sklearn/skops artifact.
simple_model_path = simple_workspace / "classifier.skops"
simple_trainer.export_model(simple_model_path)

### Load and predict

`load_skops` returns the calibrated sklearn estimator, not an inference wrapper. We materialize only the held-out real-data split and compare the loaded estimator with the trainer's fitted estimator.

In [ ]:
from GalaxySpectrumClassifier import to_xy

simple_features, simple_labels = to_xy(simple_test)
simple_model = load_skops(simple_model_path)

# Round-trip predictions must match on the held-out Cloudy rows.
np.testing.assert_array_equal(
    simple_model.predict(simple_features),
    simple_trainer.model.predict(simple_features),
)
simple_model.predict_proba(simple_features[:3])

## EpochTrainer and the epoch loaders

This section reads the existing gold-level train, validation, and test splits also used by `epochtrainer_examples.ipynb`. An epoch export contains `model.yaml` plus `params.pt` for the default format or `model.pt` for the PyTorch format.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import torch
import yaml

from GalaxySpectrumClassifier import EpochTrainer, load_default, load_torch


def find_binary_project_root():
    """Find the checked-out project and its real example data."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/gold/epoch_trainer_example").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the epoch example data")


def to_binary_float_labels(batch):
    """Give BCEWithLogitsLoss floating-point encoded 0/1 targets."""
    batch = dict(batch)
    batch["source"] = [float(value) for value in batch["source"]]
    return batch


binary_root = find_binary_project_root()
with (binary_root / "configs/binary_classifier_epoch_example.yaml").open() as stream:
    binary_config = yaml.safe_load(stream)["trainer"]

# Replace notebook-relative paths and the __main__ transform reference.
binary_data_root = binary_root / "data/gold/epoch_trainer_example"
for split in ("train", "val", "test"):
    binary_config[f"{split}_dataset_args"] = [str(binary_data_root / split)]
    binary_config[f"{split}_dataset_kwargs"]["transform"] = to_binary_float_labels

### Configure and train the binary model

The model architecture and optimizer still come from the real-data example config. We reduce training to one epoch, disable progress rendering, and redirect outputs to a temporary directory so rerunning the notebook is safe.

In [ ]:
binary_workspace = Path(tempfile.mkdtemp(prefix="gsc-binary-inference-"))
binary_config["output_path"] = str(binary_workspace / "training")
binary_config["max_epochs"] = 1
binary_config["progressbar"] = False

binary_trainer = EpochTrainer.from_config(binary_config)
binary_trainer.train()

# Use real held-out test features for all prediction comparisons below.
binary_features = torch.stack(
    [binary_trainer.eval_ds[index][0] for index in range(32)]
).numpy()

### Export, load, and compare formats

The default export writes skorch parameters to `params.pt`. The `pt` export writes the same trained module state to `model.pt`. Both loaders reconstruct native skorch estimators from the manifest and selected weight file.

In [ ]:
binary_trainer.export_model("default-export")
binary_trainer.config["export_format"] = "pt"
binary_trainer.export_model("pt-export")

binary_default_model = load_default(binary_workspace / "training/default-export")
binary_torch_model = load_torch(binary_workspace / "training/pt-export")
binary_expected = binary_trainer.model.predict(binary_features)

# Serialization format must not change predictions on the real test rows.
np.testing.assert_array_equal(
    binary_default_model.predict(binary_features), binary_expected
)
np.testing.assert_array_equal(
    binary_torch_model.predict(binary_features), binary_expected
)
binary_default_model.predict_proba(binary_features[:3])

## Class-count reconstruction with `nclasses`

The available Cloudy example has two real classes. We train the multiclass-capable `NeuralNetClassifier` on those AGN/POPSTAR labels with `nclasses=2`; the same loading contract applies unchanged to datasets with three or more encoded classes.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import torch
import yaml

from GalaxySpectrumClassifier import EpochTrainer, load_default


def find_multiclass_project_root():
    """Find the real gold-level AGN/POPSTAR dataset."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/gold/epoch_trainer_example").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the epoch example data")


multi_root = find_multiclass_project_root()
with (multi_root / "configs/binary_classifier_epoch_example.yaml").open() as stream:
    multi_config = yaml.safe_load(stream)["trainer"]

# Reuse the real split files, whose source column already contains int labels.
multi_data_root = multi_root / "data/gold/epoch_trainer_example"
for split in ("train", "val", "test"):
    multi_config[f"{split}_dataset_args"] = [str(multi_data_root / split)]
    multi_config[f"{split}_dataset_kwargs"].pop("transform", None)

### Train a multiclass-capable estimator

Switching to `CrossEntropyLoss`, two output logits, and `task="multiclass-classification"` selects skorch's `NeuralNetClassifier`. The underlying observations and encoded labels remain the real two-class Cloudy data.

In [ ]:
multi_workspace = Path(tempfile.mkdtemp(prefix="gsc-class-count-inference-"))
multi_config["output_path"] = str(multi_workspace / "training")
multi_config["max_epochs"] = 1
multi_config["progressbar"] = False
multi_config["task"] = "multiclass-classification"
multi_config["loss_type"] = "torch.nn.CrossEntropyLoss"
multi_config["model_kwargs"]["hidden_channels"] = [64, 2]
multi_config["nclasses"] = 2

multi_trainer = EpochTrainer.from_config(multi_config)
multi_trainer.train()
multi_trainer.export_model("default-export")
multi_features = torch.stack(
    [multi_trainer.eval_ds[index][0] for index in range(32)]
).numpy()

### Reconstruct the encoded class range

An epoch export records the model and loss configuration, but the class count is supplied explicitly when loading. Here the reconstructed estimator should expose the two real encoded classes `[0, 1]` and preserve predictions.

In [ ]:
multi_model = load_default(multi_workspace / "training/default-export", nclasses=2)

# Verify class metadata and predictions on held-out Cloudy rows.
np.testing.assert_array_equal(multi_model.classes_, np.arange(2))
np.testing.assert_array_equal(
    multi_model.predict(multi_features),
    multi_trainer.model.predict(multi_features),
)
multi_model.predict_proba(multi_features[:3])